# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue is a triage list, not an instruction to edit automatically. Start with the highest-ranked pages, read the reason codes, and confirm the underlying signal before choosing an action. `refresh_and_review_ctr` means declining or model-risk signals coincide with a visible page that may merit a click-through review; `refresh_and_review_engagement` asks an editor to inspect engagement signals; `expand_and_refresh` flags a thin, visible page; `refresh` is a broader review suggestion; and `monitor` means the current evidence does not justify a stronger action. Reason codes describe observed signals and may overlap; they do not prove why performance changed.

In the current 30,000-row queue, the suggested-action mix is 13,069 monitor, 8,207 refresh, 6,655 refresh-and-review-CTR, 1,987 refresh-and-review-engagement, and 82 expand-and-refresh. The top 20 contain 17 CTR reviews, two refreshes, and one engagement review. This is a prioritization view for an editor, not evidence that any action will definitely improve the traffic.

In [1]:
from pathlib import Path
import sys

import pandas as pd

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "scripts" / "ml_utils.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing scripts/ml_utils.py.")

queue_path = repo_root / "outputs" / "refresh_queue.csv"
if not queue_path.is_file():
    raise FileNotFoundError(
        f"Missing {queue_path}. Run `py scripts/run_all.py` from the repository root first."
    )

required_columns = {
    "final_rank",
    "final_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "best_model_probability",
    "content_id",
}
queue = pd.read_csv(queue_path)
missing_columns = sorted(required_columns - set(queue.columns))
if missing_columns:
    raise ValueError(f"The generated queue is missing required columns: {missing_columns}")
queue = queue.sort_values("final_rank").reset_index(drop=True)

action_order = [
    "refresh_and_review_ctr",
    "refresh_and_review_engagement",
    "expand_and_refresh",
    "refresh",
    "monitor",
]
action_summary = queue["suggested_action"].value_counts().reindex(action_order, fill_value=0)
confidence_summary = queue["confidence"].value_counts().reindex(["high", "medium", "low"], fill_value=0)
top20 = queue.head(20)
top20_action_summary = top20["suggested_action"].value_counts().reindex(action_order, fill_value=0)
reason_counts = (
    queue["final_reason_codes"].fillna("").astype(str).str.split("|").explode()
    .replace("", pd.NA).dropna().value_counts()
)

print(f"Ranked items: {len(queue):,}")
print("\nSuggested actions (counts):")
print(action_summary.to_string())
print("\nConfidence labels (counts; these are rule-based buckets):")
print(confidence_summary.to_string())
print("\nTop-20 action mix:")
print(top20_action_summary[top20_action_summary > 0].to_string())
print("\nMost frequent reason codes (non-exclusive counts):")
print(reason_counts.head(8).to_string())

Ranked items: 30,000

Suggested actions (counts):
suggested_action
refresh_and_review_ctr            6655
refresh_and_review_engagement     1987
expand_and_refresh                  82
refresh                           8207
monitor                          13069

Confidence labels (counts; these are rule-based buckets):
confidence
high       3576
medium    11424
low       15000

Top-20 action mix:
suggested_action
refresh_and_review_ctr           17
refresh_and_review_engagement     1
refresh                           2

Most frequent reason codes (non-exclusive counts):
final_reason_codes
declining_with_demand          13152
visible_model_opportunity      10653
low_ctr_visible_page            9759
ctr_review_candidate            9759
general_refresh_review          9117
model_decline_risk              8459
page_one_decay_risk             7076
low_engagement_visible_page     6508


## 2. Intended use and limits

The intended user is an SEO or editorial reviewer deciding which existing pages to should be inspected first. `final_refresh_score` blends model probability with a normalized rule score, so it is a ranking score, not a calibrated probability of success (it doesn't promise success). The queue's `confidence` label is also a rule-based screening bucket, not a guarantee.

Use the list to focus a human review. Do not interpret it as a causal estimate, a promise of traffic recovery, or a forecast of future Google rankings, because eventually we do need a human reviewer. The underlying data is a single rolling snapshot; the target compares recent and prior 30-day trends, while several features summarize the same 90-day window. Client-holdout validation tests transfer to unseen clients in this snapshot, not future-time performance. The queue also does not measure editorial quality, technical causes, revenue, or whether a proposed change will work.

In [2]:
intended_use_limits = pd.DataFrame(
    [
        {"topic": "Intended user", "guidance": "SEO or editorial reviewer prioritizing a manual page review."},
        {"topic": "Supported decision", "guidance": "Which existing items to inspect first and which observable reason to verify."},
        {"topic": "Score meaning", "guidance": "Blended ranking score; not a calibrated probability or guaranteed outcome."},
        {"topic": "Validation boundary", "guidance": "Client holdout on one snapshot; not a future-time evaluation."},
        {"topic": "Not supported", "guidance": "Automatic edits, causal impact claims, revenue estimates, or ranking guarantees."},
    ]
)
print(intended_use_limits.to_string(index=False))
print(f"\nQueue score range: {queue['final_refresh_score'].min():.1f} to {queue['final_refresh_score'].max():.1f}")
print("Model-probability values are inputs to the blended score, not probabilities that an editorial action will succeed.")

              topic                                                                         guidance
      Intended user                     SEO or editorial reviewer prioritizing a manual page review.
 Supported decision     Which existing items to inspect first and which observable reason to verify.
      Score meaning       Blended ranking score; not a calibrated probability or guaranteed outcome.
Validation boundary                    Client holdout on one snapshot; not a future-time evaluation.
      Not supported Automatic edits, causal impact claims, revenue estimates, or ranking guarantees.

Queue score range: 0.8 to 81.9
Model-probability values are inputs to the blended score, not probabilities that an editorial action will succeed.


## 3. Human review + the no-go list

Before acting, a reviewer must confirm that the page still exists and the snapshot is current; check the trend, impressions, position, CTR, freshness, and reason codes against the live page; consider seasonality, technical/indexing issues, and changes in search intent; then record an editor's decision and rationale. If the evidence is stale, contradictory, or missing, defer the recommendation and investigate further rather than forcing an action.

**No-go:** never auto-publish, rewrite, redirect, merge, or delete a page from this score; never promise a ranking or traffic gain; and never use this queue to evaluate people or clients. Eventually a human owns the decision.

In [3]:
human_review_checks = [
    "Confirm the page and snapshot are current; check whether it was already updated.",
    "Verify the reason codes against current impressions, position, CTR, engagement, and page context.",
    "Check for seasonality, technical/indexing problems, intent changes, and measurement gaps.",
    "Choose an action only after editorial review; record the decision and why it differs, if applicable.",
    "Defer if evidence is stale, contradictory, or missing.",
]
never_automate = [
    "Publishing or rewriting content",
    "Redirecting, merging, or deleting a page",
    "Guaranteeing traffic, revenue, or ranking outcomes",
    "Using the score to evaluate people or clients",
]
print("Human review checklist:")
for check in human_review_checks:
    print(f"- {check}")
print("\nNever automate:")
for action in never_automate:
    print(f"- {action}")
print("\nTop-20 items by action (no page IDs displayed):")
print(top20_action_summary[top20_action_summary > 0].to_string())

Human review checklist:
- Confirm the page and snapshot are current; check whether it was already updated.
- Verify the reason codes against current impressions, position, CTR, engagement, and page context.
- Check for seasonality, technical/indexing problems, intent changes, and measurement gaps.
- Choose an action only after editorial review; record the decision and why it differs, if applicable.
- Defer if evidence is stale, contradictory, or missing.

Never automate:
- Publishing or rewriting content
- Redirecting, merging, or deleting a page
- Guaranteeing traffic, revenue, or ranking outcomes
- Using the score to evaluate people or clients

Top-20 items by action (no page IDs displayed):
suggested_action
refresh_and_review_ctr           17
refresh_and_review_engagement     1
refresh                           2


## 4. Monitoring / retrain triggers

These are proposed operating rules, not thresholds validated by this snapshot. When a later, fully observed period is available, measure precision@20 and precision@50 beside the declining-label base rate and compare the model with the frozen rule baseline using a time-aware holdout. Review the system if precision@50 falls below that baseline for two consecutive completed periods, if the label rate moves by at least 10 percentage points from the current reference, or if an input's missingness changes by more than 5 percentage points from its training profile. A schema or definition change pauses use immediately. Investigate first; retrain only after the data issue or drift is understood and the candidate model passes a fresh honest evaluation.

In [4]:
reference_label_rate = float(queue["is_declining_label"].mean()) if "is_declining_label" in queue.columns else None
monitoring_triggers = pd.DataFrame(
    [
        {"signal": "Future ranking quality", "trigger": "Precision@50 below the frozen rule baseline for 2 consecutive fully labeled periods.", "response": "Pause expansion; investigate and rerun a time-aware evaluation."},
        {"signal": "Label prevalence", "trigger": "At least a 10 percentage-point shift from the current snapshot reference.", "response": "Check population and label-window changes before considering retraining."},
        {"signal": "Feature data quality", "trigger": "Any model-input missingness changes by more than 5 percentage points from its training profile.", "response": "Audit the pipeline and defer affected recommendations."},
        {"signal": "Schema or definition", "trigger": "A required field, metric definition, or label-window definition changes.", "response": "Stop scoring until the feature contract and validation are updated."},
    ]
)
print(monitoring_triggers.to_string(index=False))
if reference_label_rate is not None:
    print(f"\nCurrent snapshot reference declining-label rate: {reference_label_rate:.1%}")
print("The trigger values above are proposed safeguards, not empirically optimized cutoffs.")

                signal                                                                                         trigger                                                                 response
Future ranking quality            Precision@50 below the frozen rule baseline for 2 consecutive fully labeled periods.          Pause expansion; investigate and rerun a time-aware evaluation.
      Label prevalence                       At least a 10 percentage-point shift from the current snapshot reference. Check population and label-window changes before considering retraining.
  Feature data quality Any model-input missingness changes by more than 5 percentage points from its training profile.                   Audit the pipeline and defer affected recommendations.
  Schema or definition                        A required field, metric definition, or label-window definition changes.      Stop scoring until the feature contract and validation are updated.

Current snapshot reference declining-la

## 5. Exports for the paper

Write the ranked queue, aggregate summaries, and reusable charts under `work/outputs/`. The local queue export includes a pseudonymous `content_id` so a reviewer can match a recommendation to the existing internal queue; it deliberately omits `client_id` and the target label. That row-level CSV is gitignored and must remain local. Use only the aggregate summaries and figures in a public paper—never publish the row-level queue.

In [6]:
import json

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.ml_utils import simple_svg_bar_chart

work_outputs = repo_root / "work" / "outputs"
work_outputs.mkdir(parents=True, exist_ok=True)

queue_export_columns = [
    "final_rank",
    "content_id",
    "final_refresh_score",
    "best_model_name",
    "best_model_probability",
    "baseline_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "content_type",
    "main_intent",
]
missing_export_columns = sorted(set(queue_export_columns) - set(queue.columns))
if missing_export_columns:
    raise ValueError(f"Cannot export the queue; required columns are missing: {missing_export_columns}")
queue_export = queue[queue_export_columns].copy()
if {"client_id", "is_declining_label"}.intersection(queue_export.columns):
    raise AssertionError("The local action export must not contain client IDs or target labels.")

queue_output_path = work_outputs / "action_playbook_queue.csv"
queue_export.to_csv(queue_output_path, index=False)

action_export = action_summary.rename_axis("suggested_action").reset_index(name="items")
confidence_export = confidence_summary.rename_axis("confidence").reset_index(name="items")
reason_export = reason_counts.head(10).rename_axis("reason_code").reset_index(name="items")
action_export.to_csv(work_outputs / "action_counts.csv", index=False)
confidence_export.to_csv(work_outputs / "confidence_counts.csv", index=False)
reason_export.to_csv(work_outputs / "top_reason_counts.csv", index=False)

simple_svg_bar_chart(
    "Suggested action mix",
    action_export["suggested_action"].tolist(),
    action_export["items"].tolist(),
    work_outputs / "action_mix.svg",
    color="#426B69",
)
simple_svg_bar_chart(
    "Queue confidence mix",
    confidence_export["confidence"].tolist(),
    confidence_export["items"].tolist(),
    work_outputs / "confidence_mix.svg",
    color="#6F4E7C",
)
simple_svg_bar_chart(
    "Most frequent reason codes",
    reason_export["reason_code"].tolist(),
    reason_export["items"].tolist(),
    work_outputs / "top_reason_codes.svg",
    color="#8C6BB1",
)

summary_payload = {
    "rows_scored": int(len(queue)),
    "action_counts": {str(key): int(value) for key, value in action_summary.items()},
    "confidence_counts": {str(key): int(value) for key, value in confidence_summary.items()},
    "top20_action_counts": {str(key): int(value) for key, value in top20_action_summary.items() if value},
    "top_reason_counts": {str(row.reason_code): int(row.items) for row in reason_export.itertuples(index=False)},
    "publication_note": "Use aggregate summaries and charts publicly; keep the pseudonymous row-level queue local.",
}
summary_path = work_outputs / "action_playbook_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print(f"Local ranked queue saved: {queue_output_path}")
print(f"Aggregate summary saved: {summary_path}")
print("Paper-ready aggregate charts saved: action_mix.svg, confidence_mix.svg, top_reason_codes.svg")
print("The queue CSV is gitignored and contains pseudonymous content IDs; do not commit or publish it.")

Local ranked queue saved: d:\git\Assignment-1\work\outputs\action_playbook_queue.csv
Aggregate summary saved: d:\git\Assignment-1\work\outputs\action_playbook_summary.json
Paper-ready aggregate charts saved: action_mix.svg, confidence_mix.svg, top_reason_codes.svg
The queue CSV is gitignored and contains pseudonymous content IDs; do not commit or publish it.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.